In [1]:
# Importar las librerías estándar de la clase
import pandas as pd
import numpy as np

# Cargar el dataset 
df = pd.read_json('../data/raw/streaming_users_dirty.json')

# Primer vistazo a los datos
print(df.head())

   user_id  age subscription_plan  monthly_watch_time_mins   country  \
0    10000   39          Estándar                    805.8    Brasil   
1    10001   37          Estándar                   1173.4  Colombia   
2    10002   28            Básico                    401.0  Colombia   
3    10003   43            Básico                     62.4   Uruguay   
4    10004   51            Básico                    477.8      Perú   

  favorite_genre last_login_date  customer_support_tickets  
0          Crime      2025-03-04                        99  
1          Crime      2019-04-02                         2  
2          Crime      2018-04-13                         0  
3       Thriller      2021-01-31                         0  
4       Thriller      2020-09-30                         1  


In [2]:
# Crear una copia del dataset para realizar todas las transformaciones sin modificar el archivo original.

df_limpio = df.copy()

In [3]:
# Analizar la cantidad y el porcentaje de valores faltantes presentes en cada columna.

nulos = pd.DataFrame({
    "Cantidad": df_limpio.isnull().sum(),
    "Porcentaje (%)": (df_limpio.isnull().sum() / len(df_limpio) * 100).round(2)
})

nulos

,Cantidad,Porcentaje (%)
user_id,0,0.00
age,0,0.00
subscription_plan,0,0.00
monthly_watch_time_mins,193,2.37
country,0,0.00
favorite_genre,240,2.94
last_login_date,320,3.92
customer_support_tickets,0,0.00


In [4]:
# Reemplazar los valores faltantes de la variable numérica con la mediana.

df_limpio["monthly_watch_time_mins"] = df_limpio["monthly_watch_time_mins"].fillna(
    df_limpio["monthly_watch_time_mins"].median()
)

En monthly_watch_time_mins presenta 193 valores faltantes, lo que representa el 2,37 % del total de registros del dataset. Al ser un porcentaje bajo, se evita la perdida reemplazándolos por la mediana de la columna, para evitar valores atípicos. 

In [5]:
# Reemplazar los valores faltantes de la variable categórica con la moda.

df_limpio["favorite_genre"] = df_limpio["favorite_genre"].fillna(
    df_limpio["favorite_genre"].mode()[0]
)

En la variable favorite_genre presenta 240 valores faltantes, siendo el 2,94 % del total, en este caso decidí reemplazarlos por la moda, ya que permite conservar la mayor cantidad de información, y siendo una columna que habla de los géneros favoritos es preferible quedarse con lo que la mayoría de los usuarios eligen

In [6]:
# Mostrar los valores únicos de la columna.

df_limpio["last_login_date"].unique()

<StringArray>
['2025-03-04', '2019-04-02', '2018-04-13', '2021-01-31', '2020-09-30',
 '2020-07-03', '2019-07-26', '2019-02-24', '2025-08-03', '2024-02-12',
 ...
 '2020/04/11', '2021/05/09', '14-03-2019', '2023-06-16', '2024-08-25',
 '2020-09-03', '2023/06/27', '2018-06-25', '2019/04/30', '2025-07-19']
Length: 3063, dtype: str

In [7]:
# Convertir la columna de fechas a formato datetime, aceptando distintos formatos.

df_limpio["last_login_date"] = pd.to_datetime(
    df_limpio["last_login_date"],
    format="mixed",
    errors="coerce"
)

En last_login_date no solo había valores nulos sino también había fechas con distintos formatos asi que se decidió cambiar solo a formato datetime para tener solo datos en un solo formato y no haya errores

In [8]:
df_limpio["last_login_date"].isna().sum()

np.int64(384)

In [9]:
# Eliminar los registros que no poseen una fecha válida de último inicio de sesión.

df_limpio = df_limpio.dropna(subset=["last_login_date"])

En la variable last_login_date había inicialmente 320 valores faltantes. Al unificar los formatos de fecha, algunos registros no pudieron convertirse correctamente, aumentando a 384 valores nulos. Decidí eliminar estos registros, ya que al tratarse de la última fecha de inicio de sesión no es adecuado imputar una fecha ficticia, conservando únicamente datos válidos y consistentes para el análisis.

In [10]:
# Verificar que ya no existan valores faltantes en la columna.

df_limpio["last_login_date"].isnull().sum()

np.int64(0)

Se verifica y confirma que ya no hay valores faltantes

In [11]:
# Verificar la cantidad de registros duplicados en el dataset.

print("Cantidad de registros duplicados:")
print(df_limpio.duplicated().sum())

Cantidad de registros duplicados:
131


In [12]:
# Eliminar los registros completamente duplicados del dataset.

df_limpio = df_limpio.drop_duplicates()

In [13]:
# Verificar que ya no existan registros duplicados.

print("Cantidad de registros duplicados:")
print(df_limpio.duplicated().sum())

Cantidad de registros duplicados:
0


En el dataset se detectó que hay 131 datos duplicados, entonces decidí eliminarlos ya que no aportan información y puede afectar los resultados finales. Luego de la limpieza, el dataset quedó sin datos duplicados garantizando la calidad de los datos.

In [14]:
# Mostrar las categorías de la variable subscription_plan.

print(df_limpio["subscription_plan"].value_counts(dropna=False))

subscription_plan
Básico       3222
Estándar     2540
Premium      1425
basico         60
Basic          50
BASICO         49
básico         47
Std            46
Estándar       45
estandar       34
STANDARD       32
Premium        27
PREMIUM        24
premium        22
Premiun        22
Name: count, dtype: int64


In [15]:
# Unificar las categorías de la variable subscription_plan.

df_limpio["subscription_plan"] = df_limpio["subscription_plan"].replace({
    "basico": "Básico",
    "básico": "Básico",
    "BASICO": "Básico",
    "Basic": "Básico",

    "estandar": "Estándar",
    "STANDARD": "Estándar",
    "Std": "Estándar",

    "premium": "Premium",
    "PREMIUM": "Premium",
    "Premiun": "Premium"
})

In [16]:
# Verificación de las categorías unificadas de la variable subscription_plan.

print(df_limpio["subscription_plan"].value_counts(dropna=False))

subscription_plan
Básico       3428
Estándar     2652
Premium      1493
Estándar       45
Premium        27
Name: count, dtype: int64


En la columna subscription_plan había muchos datos duplicados, así que decidí unificarlos para evitar problemas futuros, luego de unificar los datos de la columna 'subscription_plan' siguen habiendo datos duplicados porque hay datos con espacios, lo que quedaa hacer es eliminarlos. 

In [17]:
# Eliminar espacios al inicio y al final de los nombres de los planes.

df_limpio["subscription_plan"] = df_limpio["subscription_plan"].str.strip()

In [18]:
print(df_limpio["subscription_plan"].value_counts(dropna=False))

subscription_plan
Básico      3428
Estándar    2697
Premium     1520
Name: count, dtype: int64


Al eliminarlos, se aseguró que ya no haya datos duplicados en la columna subscription_plan 

In [19]:
# Mostrar las categorías de la variable country.

print(df_limpio["country"].value_counts(dropna=False).to_string())

country
Chile         1067
Brasil        1062
Colombia      1053
Uruguay       1049
México        1046
Perú          1041
Argentina     1028
colombia        27
uruguay         23
méxico          22
Brazil          20
COL             19
URY             16
chile           15
Mexico          15
Chile           15
Peru            15
argentina       15
PER             15
CHL             15
brasil          13
MEX             13
BRA             13
ARG             10
perú             9
Argentina        9


In [20]:
# Unificar las categorías de la variable country.

df_limpio["country"] = df_limpio["country"].replace({
    "argentina": "Argentina",
    "ARG": "Argentina",

    "brasil": "Brasil",
    "Brazil": "Brasil",
    "BRA": "Brasil",

    "colombia": "Colombia",
    "COL": "Colombia",

    "uruguay": "Uruguay",
    "URY": "Uruguay",

    "méxico": "México",
    "Mexico": "México",
    "MEX": "México",

    "perú": "Perú",
    "Peru": "Perú",
    "PER": "Perú",

    "chile": "Chile",
    "CHL": "Chile"
})

# Eliminar espacios al inicio y al final.
df_limpio["country"] = df_limpio["country"].str.strip()

In [21]:
# Verificar las categorías luego de la limpieza.

print(df_limpio["country"].value_counts(dropna=False))

country
Chile        1112
Brasil       1108
Colombia     1099
México       1096
Uruguay      1088
Perú         1080
Argentina    1062
Name: count, dtype: int64


Ahora se trabaja con la columna country se ve que hay varios datos duplicados, así que tomé la decisión de unificarlos y eliminar los espacios, luego verifiqué y todo quedó sin duplicados.

In [22]:
# Mostrar las categorías de la variable favorite_genre.

print(df_limpio["favorite_genre"].value_counts(dropna=False).to_string())

favorite_genre
Comedia        1277
Drama          1031
Thriller       1029
Documental     1018
Acción         1016
Romance        1014
Crime           991
Action           20
COMEDIA          18
CRIME            17
Crimen           17
DRAMA            16
Romance          16
Comedia          15
Documentary      15
DOC              15
ACCIÓN           14
THRILLER         14
ROMANCE          13
comedy           12
Thriller         12
romance          12
documental       10
drama             9
accion            8
Drama             7
thriler           5
crime             4


In [23]:
# Unificar las categorías de la variable favorite_genre.

df_limpio["favorite_genre"] = df_limpio["favorite_genre"].replace({
    "Action": "Acción",
    "ACCIÓN": "Acción",
    "accion": "Acción",

    "COMEDIA": "Comedia",
    "comedy": "Comedia",

    "DRAMA": "Drama",
    "drama": "Drama",

    "CRIME": "Crime",
    "crime": "Crime",
    "Crimen": "Crime",

    "Documentary": "Documental",
    "DOC": "Documental",
    "documental": "Documental",

    "THRILLER": "Thriller",
    "thriler": "Thriller",

    "ROMANCE": "Romance",
    "romance": "Romance"
})

# Eliminar espacios al inicio y al final.
df_limpio["favorite_genre"] = df_limpio["favorite_genre"].str.strip()

In [24]:
# Verificar las categorías luego de la limpieza.

print(df_limpio["favorite_genre"].value_counts(dropna=False))

favorite_genre
Comedia       1322
Drama         1063
Thriller      1060
Acción        1058
Documental    1058
Romance       1055
Crime         1029
Name: count, dtype: int64


En la última columna categórita que es la variable favorite_genre, había varios valores duplicados y nombrados de diferentes formas, hice la limpieza como las anteriores, unifiqué los datos duplicados y eliminé los espacios, dejando así la columna libre de duplicados, lo cuál se ve en la celda de código de verificación

In [25]:
# Detectar la cantidad de valores atípicos utilizando el método del rango intercuartílico (IQR).

columnas_numericas = ["age", "monthly_watch_time_mins", "customer_support_tickets"]

for columna in columnas_numericas:

    Q1 = df_limpio[columna].quantile(0.25)
    Q3 = df_limpio[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    cantidad = ((df_limpio[columna] < limite_inferior) |
                (df_limpio[columna] > limite_superior)).sum()

    print(f"{columna}: {cantidad} valores atípicos")

age: 86 valores atípicos
monthly_watch_time_mins: 147 valores atípicos
customer_support_tickets: 356 valores atípicos


Para saber si hay valores atípicos analicé las variables numéricas con (IQR) para identificarlos

In [26]:
# Mostrar estadísticas de las variables numéricas.

print(df_limpio[["age", "monthly_watch_time_mins", "customer_support_tickets"]].describe())

               age  monthly_watch_time_mins  customer_support_tickets
count  7645.000000              7645.000000               7645.000000
mean     34.054545              1082.054179                  1.797122
std      14.556995              5053.065749                 11.654634
min      -5.000000              -120.000000                 -1.000000
25%      25.000000               495.600000                  0.000000
50%      33.000000               757.400000                  1.000000
75%      42.000000              1036.800000                  1.000000
max     150.000000             99999.000000                150.000000


In [27]:
# Contar la cantidad de registros con valores fuera de los rangos esperados.

print("Edad menor que 0:", (df_limpio["age"] < 0).sum())
print("Edad mayor que 100:", (df_limpio["age"] > 100).sum())

print("Tiempo de visualización menor que 0:",
      (df_limpio["monthly_watch_time_mins"] < 0).sum())
print("Tiempo de visualización mayor que 30000:",
      (df_limpio["monthly_watch_time_mins"] > 30000).sum())

print("Tickets menores que 0:",
      (df_limpio["customer_support_tickets"] < 0).sum())
print("Tickets mayores que 50:",
      (df_limpio["customer_support_tickets"] > 50).sum())

Edad menor que 0: 20
Edad mayor que 100: 50
Tiempo de visualización menor que 0: 47
Tiempo de visualización mayor que 30000: 28
Tickets menores que 0: 27
Tickets mayores que 50: 66


Antes de modificar o eliminar los valores atípicos, contabilicé los registros con valores que no son lógicos ni coherentes. Se detectó muchos valores fuera de rangos lógicos, por ejemplo, se encontró que hay 20 datos de edades negativas, lo que es imposible, además, 47 datos de tiempo de visualización negativos. Estos resultados demuestran la presencia de datos inconsistentes

In [28]:
# Eliminar registros con valores fuera de rangos válidos.

df_limpio = df_limpio[
    (df_limpio["age"] >= 0) &
    (df_limpio["age"] <= 90) &
    (df_limpio["monthly_watch_time_mins"] >= 0) &
    (df_limpio["monthly_watch_time_mins"] <= 5000) &
    (df_limpio["customer_support_tickets"] >= 0) &
    (df_limpio["customer_support_tickets"] <= 20)
]

print(df_limpio[["age", "monthly_watch_time_mins", "customer_support_tickets"]].describe().round(2))

           age  monthly_watch_time_mins  customer_support_tickets
count  7409.00                  7409.00                   7409.00
mean     33.47                   796.94                      0.74
std      11.79                   492.97                      0.86
min       0.00                     0.00                      0.00
25%      25.00                   500.40                      0.00
50%      33.00                   757.40                      1.00
75%      41.00                  1035.50                      1.00
max      80.00                  4193.70                      5.00


Decidí eliminar los registros que contenían datos imposibles para conservar solamente registros coherentes, mejorando la calidad del dataset. En la variable age eliminé las edades negativas y las superiores a 90 años, ya que representan valores inconsistentes. Aunque un niño pequeño puede consumir contenido en una plataforma de streaming, normalmente lo hace utilizando la cuenta de sus padres o tutores, por lo que decidí conservar todas las edades válidas y eliminar únicamente los valores claramente imposibles.

In [29]:
# Guardar el dataset limpio para utilizarlo en las siguientes etapas del proyecto.

df_limpio.to_csv("../data/processed/streaming_dataset_limpio.csv", index=False)

In [30]:
# GENERACIÓN AUTOMÁTICA DEL LOG ETL PASO A PASO (REQUERIMIENTO OBLIGATORIO)
import os
import pandas as pd

# Variables de control basadas en el tamaño inicial
filas_iniciales = len(df)
log_pasos = []

# --- PASO 0: Estado Inicial ---
df_p0 = df.copy()
log_pasos.append({
    "Paso": 0,
    "Descripción": "Estado Inicial del Dataset (Raw)",
    "Filas": len(df_p0),
    "Nulos": df_p0.isnull().sum().sum(),
    "Retención (%)": 100.0
})

# --- PASO 1: Imputación de Valores Faltantes ---
df_p1 = df_p0.copy()
df_p1["monthly_watch_time_mins"] = df_p1["monthly_watch_time_mins"].fillna(df_p1["monthly_watch_time_mins"].median())
df_p1["favorite_genre"] = df_p1["favorite_genre"].fillna(df_p1["favorite_genre"].mode()[0])
log_pasos.append({
    "Paso": 1,
    "Descripción": "Imputación de nulos (Mediana en watch_time y Moda en favorite_genre)",
    "Filas": len(df_p1),
    "Nulos": df_p1.isnull().sum().sum(),
    "Retención (%)": round((len(df_p1) / filas_iniciales) * 100, 2)
})

# --- PASO 2: Limpieza de Fechas ---
df_p2 = df_p1.copy()
df_p2["last_login_date"] = pd.to_datetime(df_p2["last_login_date"], format="mixed", errors="coerce")
df_p2 = df_p2.dropna(subset=["last_login_date"])
log_pasos.append({
    "Paso": 2,
    "Descripción": "Eliminación de registros con fechas inválidas o nulas en last_login_date",
    "Filas": len(df_p2),
    "Nulos": df_p2.isnull().sum().sum(),
    "Retención (%)": round((len(df_p2) / filas_iniciales) * 100, 2)
})

# --- PASO 3: Eliminación de Duplicados ---
df_p3 = df_p2.copy()
df_p3 = df_p3.drop_duplicates()
log_pasos.append({
    "Paso": 3,
    "Descripción": "Eliminación de registros completamente duplicados",
    "Filas": len(df_p3),
    "Nulos": df_p3.isnull().sum().sum(),
    "Retención (%)": round((len(df_p3) / filas_iniciales) * 100, 2)
})

# --- PASO 4: Normalización de Variables Categóricas ---
df_p4 = df_p3.copy()
# (Se simula el proceso de stripping y reemplazo para medir nulos resultantes si los hubiera)
df_p4["subscription_plan"] = df_p4["subscription_plan"].str.strip()
df_p4["country"] = df_p4["country"].str.strip()
df_p4["favorite_genre"] = df_p4["favorite_genre"].str.strip()
log_pasos.append({
    "Paso": 4,
    "Descripción": "Normalización de texto y unificación de categorías (Suscripción, País, Género)",
    "Filas": len(df_p4),
    "Nulos": df_p4.isnull().sum().sum(),
    "Retención (%)": round((len(df_p4) / filas_iniciales) * 100, 2)
})

# --- PASO 5: Filtro de Outliers e Inconsistencias (Estado Final) ---
# Usamos directamente tu df_limpio final que ya pasó por este filtro en tu código anterior
log_pasos.append({
    "Paso": 5,
    "Descripción": "Filtro de valores incoherentes (Edades, tiempos y tickets fuera de rango)",
    "Filas": len(df_limpio),
    "Nulos": df_limpio.isnull().sum().sum(),
    "Retención (%)": round((len(df_limpio) / filas_iniciales) * 100, 2)
})

# =====================================================================
# GUARDAR Y MOSTRAR EL LOG ETL OBLIGATORIO
# =====================================================================
df_pipeline_log = pd.DataFrame(log_pasos)
df_pipeline_log.to_csv("../logs/pipeline_log.csv", index=False, encoding="utf-8")

print("✔️ ¡El archivo logs/pipeline_log.csv fue generado con el historial completo de transformaciones!")
df_pipeline_log

✔️ ¡El archivo logs/pipeline_log.csv fue generado con el historial completo de transformaciones!


,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Estado Inicial del Dataset (Raw),8160,753,100.00
1,1,Imputación de nulos (Mediana en watch_time y M...,8160,320,100.00
2,2,Eliminación de registros con fechas inválidas ...,7776,0,95.29
3,3,Eliminación de registros completamente duplicados,7645,0,93.69
4,4,Normalización de texto y unificación de catego...,7645,0,93.69
5,5,"Filtro de valores incoherentes (Edades, tiempo...",7409,0,90.80
